In [157]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load dataset

In [158]:
def load_data(file_path):
    try:
        data = pd.read_csv(file_path)
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    
df = load_data("../../Database/PDF/Customer-Churn-dataset.csv")

## Fraud logic (Rule-Based)

In [159]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

# Create fraud logic on condition
import numpy as np
conditions = (
    (df["Balance"] > 150000) &
    (df["NumOfProducts"] <=1 ) &
    (df["IsActiveMember"] == 0)
) | (
    (df["CreditScore"] < 400) &
    (df["Balance"] > 100000)
) | (
    (df["Complain"] > 0) &
    (df["Satisfaction Score"] <= 2)
)

df["Fraud"] = np.where(conditions, 1, 0)

In [160]:
# reducte fraud cases aritifically
fraud_idx = df[df["Fraud"] == 1].sample(frac=0.3, random_state=42).index
df["Fraud"] = 0
df.loc[fraud_idx, "Fraud"] = 1

## Add advanced fraud features

In [161]:
# High risk score (engineered features)
df["RiskScore"] = (
    (df["CreditScore"] < 500).astype(int) +
    (df["Balance"] > 100000).astype(int) +
    (df["IsActiveMember"] == 0).astype(int) +
    (df["Complain"] > 0).astype(int)
)

# Balance per product
df["BalancePerProduct"] = df["Balance"] / (df["NumOfProducts"] + 1)

# Age risk bucket
df["AgeRisk"] = np.where(df["Age"] < 25, 1, np.where(df["Age"] > 60, 1, 0))

In [162]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,0,1,0.000000,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,0,1,41903.930000,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,0,3,39915.200000,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,0,1,0.000000,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,0,1,62755.410000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300,0,1,0.000000,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771,0,0,28684.805000,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564,0,1,0.000000,0
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339,1,2,25025.103333,0


In [163]:
print(df["Fraud"].value_counts(normalize=True))

0    0.9668
1    0.0332
Name: Fraud, dtype: float64


## Add features for Market Risk

In [164]:
df["HighValueCustomer"] = (df["Balance"] > 100000).astype(int)
df["LowCreditRisk"] = (df["CreditScore"] < 500).astype(int)

# Mkarketing score
"""Marketing score: combines high value customer status, low credit risk, and non-France geography to identify customers 
who may be more receptive to marketing campaigns, which can help target efforts effectively."""
df["MarketingScore"] = (
    df["HighValueCustomer"] + df["LowCreditRisk"] + (df["Geography"] != "France").astype(int)
).drop(columns=["HighValueCustomer", "LowCreditRisk"])

## Operation Risk feature for internal banking

In [165]:
df["ComplainFlag"] = (df["Complain"] > 0).astype(int)
df["LowSatisfaction"] = (df["Satisfaction Score"] <= 2).astype(int)

# Systemic risk score
"""Operational risk score: combines complaint and satisfaction flags with inactivity to identify customers at risk of leaving or being dissatisfied, 
which can impact operational stability."""
df["OperationalRiskScore"] = (
    df["ComplainFlag"] +
    df["LowSatisfaction"] +
    (df["IsActiveMember"] == 0).astype(int)
).drop(columns=["ComplainFlag", "LowSatisfaction"])

In [166]:
pd.set_option("display.max_columns", None)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339,1,2,25025.103333,0,0,0,1,1,1,3


## Data encoding

In [167]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
# Encode categorical features
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded {col}")

Encoded Geography
Encoded Gender
Encoded Card Type


In [168]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,0,0,42,2,0.00,1,1,1,101348.88,1,1,2,0,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0,1,3,0,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1,1,3,0,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,0,0,39,1,0.00,2,0,0,93826.63,0,0,5,1,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0,0,5,1,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,0,1,39,5,0.00,2,1,0,96270.64,0,0,1,0,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,0,1,35,10,57369.61,1,1,1,101699.77,0,0,5,2,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,0,0,36,7,0.00,1,0,1,42085.58,1,1,3,3,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,1,1,42,3,75075.31,2,1,0,92888.52,1,1,2,1,339,1,2,25025.103333,0,0,0,1,1,1,3


## Define features for X and y -> (Fraud detection - Market Risk - Operation Risk)

In [169]:
FEATURES = [col for col in df.columns if col not in ["Fraud", "MarketingScore", "OperationalRiskScore"]]
TARGET_FRAUD = "Fraud"
TARGET_MARKETING = "MarketingScore"
TARGET_OPERATIONAL = "OperationalRiskScore"

# Define X and y for each target
X = df[FEATURES]
y_fraud = df[TARGET_FRAUD]
y_marketing = df[TARGET_MARKETING]
y_operational = df[TARGET_OPERATIONAL]

from sklearn.model_selection import train_test_split
# Clean code for splitting data into train and test sets for each target variable
def split_data(X, y, test_size=0.2, random_state=42, stratify=None):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=stratify)

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = split_data(X, y_fraud, stratify=y_fraud) # Split data for fraud prediction
X_train_marketing, X_test_marketing, y_train_marketing, y_test_marketing = split_data(X, y_marketing, stratify=y_marketing) # Split data for marketing score prediction
X_train_operational, X_test_operational, y_train_operational, y_test_operational = split_data(X, y_operational, stratify=y_operational) # Split data for operational risk score prediction

## Data scaling

In [170]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# Scale features for fraud detection
X_train_fraud_scaled = scaler.fit_transform(X_train_fraud)
X_test_fraud_scaled = scaler.transform(X_test_fraud)
# Scale features for marketing score prediction
X_train_marketing_scaled = scaler.fit_transform(X_train_marketing)
X_test_marketing_scaled = scaler.transform(X_test_marketing)
# Scale features for operational risk score prediction
X_train_operational_scaled = scaler.fit_transform(X_train_operational)
X_test_operational_scaled = scaler.transform(X_test_operational)

print(f"Data scaled for fraud detection: {X_train_fraud_scaled.shape}, {X_test_fraud_scaled.shape}")
print(f"Data scaled for marketing score prediction: {X_train_marketing_scaled.shape}, {X_test_marketing_scaled.shape}")
print(f"Data scaled for operational risk score prediction: {X_train_operational_scaled.shape}, {X_test_operational_scaled.shape}")

Data scaled for fraud detection: (8000, 22), (2000, 22)
Data scaled for marketing score prediction: (8000, 22), (2000, 22)
Data scaled for operational risk score prediction: (8000, 22), (2000, 22)


## Machine learning models

In [171]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore")

# Define models
model = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0),
    "KNN": KNeighborsClassifier()
}

# Define hyperparameters for tuning
param_grid = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "penalty": ["l2"],
        "solver": ["lbfgs"]
    },
    "Decision Tree": {
        "max_depth": [10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [10, 20],
        "min_samples_split": [2, 5]
    },
    "XGBoost": {
        "n_estimators": [100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7],
        "weights": ["uniform", "distance"]
    }
}

def tune_model(model, param_grid, X_train, y_train):
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=5,
        cv=3,
        verbose=0,
        random_state=42,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)
    return random_search.best_estimator_

# Function to get appropriate average for metrics based on number of classes
def get_average(y_true):
    n_classes = len(set(y_true))
    return 'binary' if n_classes == 2 else 'weighted'

# Store results
all_results = {}

# Train and evaluate models
for name, clf in model.items():
    print(f"\n{'='*70}")
    print(f"Training {name}...")
    print(f"{'='*70}")
    
    results = {}
    
    # FRAUD DETECTION (Binary)
    print(f"\n--- Fraud Detection ---")
    best_clf_fraud = tune_model(clf, param_grid[name], X_train_fraud_scaled, y_train_fraud)
    y_pred_fraud = best_clf_fraud.predict(X_test_fraud_scaled)
    avg_fraud = get_average(y_test_fraud)
    
    results['Fraud'] = {
        "Accuracy": accuracy_score(y_test_fraud, y_pred_fraud),
        "Precision": precision_score(y_test_fraud, y_pred_fraud, average=avg_fraud),
        "Recall": recall_score(y_test_fraud, y_pred_fraud, average=avg_fraud),
        "F1 Score": f1_score(y_test_fraud, y_pred_fraud, average=avg_fraud),
        "ROC AUC": roc_auc_score(y_test_fraud, best_clf_fraud.predict_proba(X_test_fraud_scaled)[:, 1])
    }
    
    print(f"Accuracy: {results['Fraud']['Accuracy']:.4f}")
    print(f"Precision: {results['Fraud']['Precision']:.4f}")
    print(f"Recall: {results['Fraud']['Recall']:.4f}")
    print(f"F1 Score: {results['Fraud']['F1 Score']:.4f}")
    print(f"ROC AUC: {results['Fraud']['ROC AUC']:.4f}")
    
    # MARKETING SCORE PREDICTION (Multiclass)
    print(f"\n--- Marketing Score Prediction ---")
    best_clf_marketing = tune_model(clf, param_grid[name], X_train_marketing_scaled, y_train_marketing)
    y_pred_marketing = best_clf_marketing.predict(X_test_marketing_scaled)
    avg_marketing = get_average(y_test_marketing)
    
    results['Marketing'] = {
        "Accuracy": accuracy_score(y_test_marketing, y_pred_marketing),
        "Precision": precision_score(y_test_marketing, y_pred_marketing, average=avg_marketing),
        "Recall": recall_score(y_test_marketing, y_pred_marketing, average=avg_marketing),
        "F1 Score": f1_score(y_test_marketing, y_pred_marketing, average=avg_marketing)
    }
    
    print(f"Accuracy: {results['Marketing']['Accuracy']:.4f}")
    print(f"Precision: {results['Marketing']['Precision']:.4f}")
    print(f"Recall: {results['Marketing']['Recall']:.4f}")
    print(f"F1 Score: {results['Marketing']['F1 Score']:.4f}")
    
    # OPERATIONAL RISK SCORE PREDICTION (Multiclass)
    print(f"\n--- Operational Risk Score Prediction ---")
    best_clf_operational = tune_model(clf, param_grid[name], X_train_operational_scaled, y_train_operational)
    y_pred_operational = best_clf_operational.predict(X_test_operational_scaled)
    avg_operational = get_average(y_test_operational)
    
    results['Operational'] = {
        "Accuracy": accuracy_score(y_test_operational, y_pred_operational),
        "Precision": precision_score(y_test_operational, y_pred_operational, average=avg_operational),
        "Recall": recall_score(y_test_operational, y_pred_operational, average=avg_operational),
        "F1 Score": f1_score(y_test_operational, y_pred_operational, average=avg_operational)
    }
    
    print(f"Accuracy: {results['Operational']['Accuracy']:.4f}")
    print(f"Precision: {results['Operational']['Precision']:.4f}")
    print(f"Recall: {results['Operational']['Recall']:.4f}")
    print(f"F1 Score: {results['Operational']['F1 Score']:.4f}")
    
    all_results[name] = results

print(f"\n{'='*70}")
print("MODEL TRAINING COMPLETE")
print(f"{'='*70}")


Training Logistic Regression...

--- Fraud Detection ---
Accuracy: 0.9670
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
ROC AUC: 0.9402

--- Marketing Score Prediction ---
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

--- Operational Risk Score Prediction ---
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

Training Decision Tree...

--- Fraud Detection ---
Accuracy: 0.9600
Precision: 0.3158
Recall: 0.1818
F1 Score: 0.2308
ROC AUC: 0.8016

--- Marketing Score Prediction ---
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

--- Operational Risk Score Prediction ---
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

Training Random Forest...

--- Fraud Detection ---
Accuracy: 0.9670
Precision: 0.5000
Recall: 0.0152
F1 Score: 0.0294
ROC AUC: 0.9720

--- Marketing Score Prediction ---
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

--- Operational Risk Score Prediction ---
Accuracy: 1.0000
